# 16 - Lakehouse Delta Maintenance

Dry-run-first maintenance for demo-owned Delta tables. It records table size/file metadata, optionally enables compatible Delta write properties, executes `OPTIMIZE` only when supported, and vacuums with a retention-safe default. Unsupported commands are logged as such and never reported as success.

In [ ]:
# PARAMETERS
artifact_prefixes = ['bronze_', 'dim_', 'bridge_', 'fact_', 'gold_', 'agent_context']
dry_run = True
apply_table_properties = True
apply_optimize = False
apply_vacuum = False
vacuum_retention_hours = 168
strict_mode = False

from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import Row

assert vacuum_retention_hours >= 168, 'Retention below seven days requires a separate governed exception and is disabled here'
RESULTS = []


def record(table_name, operation, status, detail=''):
    row = Row(
        table_name=table_name, operation=operation, status=status,
        detail=str(detail)[:4000], observed_at=datetime.now(timezone.utc), is_synthetic=True)
    RESULTS.append(row)
    print(status, operation, table_name, detail)


def is_owned_table(table_name):
    return any(table_name == prefix or table_name.startswith(prefix) for prefix in artifact_prefixes)


table_names = sorted(
    table.name for table in spark.catalog.listTables()
    if table.tableType != 'VIEW' and is_owned_table(table.name))
assert table_names, 'No demo-owned Lakehouse tables were found'

for table_name in table_names:
    try:
        detail = spark.sql('DESCRIBE DETAIL `' + table_name.replace('`','') + '`').first().asDict()
        record(table_name, 'DESCRIBE_DETAIL', 'SUCCEEDED', 'files=' + str(detail.get('numFiles')) + ', bytes=' + str(detail.get('sizeInBytes')))
    except Exception as exc:
        record(table_name, 'DESCRIBE_DETAIL', 'FAILED', exc)
        if strict_mode:
            raise
        continue

    if apply_table_properties:
        if dry_run:
            record(table_name, 'TABLE_PROPERTIES', 'DRY_RUN', 'Would enable Delta optimizeWrite/autoCompact when supported')
        else:
            try:
                spark.sql("ALTER TABLE `" + table_name.replace('`','') + "` SET TBLPROPERTIES ('delta.autoOptimize.optimizeWrite'='true','delta.autoOptimize.autoCompact'='true')")
                record(table_name, 'TABLE_PROPERTIES', 'SUCCEEDED', 'Delta write properties set')
            except Exception as exc:
                record(table_name, 'TABLE_PROPERTIES', 'SKIPPED_UNSUPPORTED', exc)
                if strict_mode:
                    raise

    if apply_optimize:
        if dry_run:
            record(table_name, 'OPTIMIZE', 'DRY_RUN', 'Would execute OPTIMIZE')
        else:
            try:
                spark.sql('OPTIMIZE `' + table_name.replace('`','') + '`')
                record(table_name, 'OPTIMIZE', 'SUCCEEDED', 'Command completed')
            except Exception as exc:
                record(table_name, 'OPTIMIZE', 'SKIPPED_UNSUPPORTED', exc)
                if strict_mode:
                    raise

    if apply_vacuum:
        if dry_run:
            record(table_name, 'VACUUM', 'DRY_RUN', 'Would vacuum with retention=' + str(vacuum_retention_hours) + ' hours')
        else:
            try:
                DeltaTable.forName(spark, table_name).vacuum(vacuum_retention_hours)
                record(table_name, 'VACUUM', 'SUCCEEDED', 'Retention=' + str(vacuum_retention_hours) + ' hours')
            except Exception as exc:
                record(table_name, 'VACUUM', 'FAILED', exc)
                if strict_mode:
                    raise

if RESULTS:
    spark.createDataFrame(RESULTS).write.mode('append').format('delta').saveAsTable('lakehouse_maintenance_results')
failures = [row for row in RESULTS if row.status == 'FAILED']
if failures and strict_mode:
    raise AssertionError('Lakehouse maintenance failures: ' + str(len(failures)))
print('Lakehouse maintenance complete:', len(table_names), 'tables; dry_run=', dry_run)